# Strand D (missingness mechanism): what IS the unmeasured visit mass?

**Question:** Blacklight (run ~Jan 2025) failed on ~47% of unique panel
domains. The interpretation of the coverage bounds (strand B) depends on what
that missing mass actually is:

1. **Tracker infrastructure, not websites.** Exploration of the never-measured
   remainder showed many "visited domains" are ad-tech endpoints (prebid/CDN/
   cookie-sync hosts) with no homepage for *any* scanner — the meter logged
   redirect hops and sync requests as visits. This cell quantifies that
   composition against the pinned Tracker Radar list. Visits to such domains
   are arguably direct evidence of tracking events, and imputing them "typical
   website tracking" (strand B's mean/p90 scenarios) is the wrong model — the
   composition split lets readers weight the scenarios accordingly.
2. **Dead vs unscannable websites.** For the rest: if domains died between
   June 2022 and the scan, the missing mass is real 2022 browsing a 2025 scan
   can never recover (compounding the temporal problem); if they were alive
   and merely unscannable (bot blocking, timeouts), contemporaneous fills are
   the right remedy. **Evidence of being alive in mid-2022**, per domain: a
   June-2022 HTTP Archive crawl row (CrUX-ranked, so definitely live), or a
   Wayback capture in the Apr-Aug 2022 window. A Wayback *miss* is not proof
   of death — long-tail coverage is incomplete — so the alive share is a
   lower bound.

In [1]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

import config
from utilities import pandas_to_tex, save_mpl_fig


Checking that all paths exist:
{'web_mobile': False, 'web_desktop': False, 'web': False, 'yg_profile': False, 'blacklight': False, 'who': False}


In [2]:
yg = pd.read_csv(config.FP_YG_IND_DOMAIN)
yg["filename"] = yg["private_domain"].str.replace(".", "_", regex=False)
bl_files = set(pd.read_csv(config.FP_BLACKLIGHT)["filename"])
ha22_domains = set(
    pd.read_csv(config.FP_HA_DOMAIN_MEASURES)
    .query("crawl == 'panel'")["private_domain"].unique()
)
manifest = pd.read_csv(config.FP_WB_MANIFEST)
wb_alive = set(manifest.loc[manifest["cdx_hit"] == True, "private_domain"])
wb_checked = set(manifest["private_domain"])

dom = (
    yg.groupby(["private_domain", "filename"])["visits"].sum().reset_index()
)
dom["bl_scanned"] = dom["filename"].isin(bl_files)
dom["ha22"] = dom["private_domain"].isin(ha22_domains)
dom["wb_hit"] = dom["private_domain"].isin(wb_alive)
dom["wb_checked"] = dom["private_domain"].isin(wb_checked)
dom["alive_2022"] = dom["ha22"] | dom["wb_hit"]

unscanned = dom.query("not bl_scanned")
print(f"BL-unscanned domains: {len(unscanned):,} "
      f"({100 * unscanned['visits'].sum() / dom['visits'].sum():.1f}% of visits)")

BL-unscanned domains: 29,996 (23.6% of visits)


## Composition: how much of the unmeasured mass is tracker infrastructure?

In [3]:
cats = pd.read_csv(config.FP_DDG_CATEGORIES)
known_trackers = set(cats["domain"])

never = unscanned.query("not ha22")  # measured by neither BL nor HA-2022
never = never.assign(is_tracker_domain=never["private_domain"].isin(known_trackers))
nv = never["visits"].sum()
comp = pd.Series({
    "never-measured domains": len(never),
    "share of all visits (%)": 100 * nv / dom["visits"].sum(),
    "tracker-infrastructure domains (%)": 100 * never["is_tracker_domain"].mean(),
    "tracker-infrastructure visit share (%)":
        100 * never.loc[never["is_tracker_domain"], "visits"].sum() / nv,
}).round(1)
print(comp.to_string())
print()
print("largest tracker-infrastructure 'visited' domains:")
print(never.query("is_tracker_domain").nlargest(8, "visits")
      [["private_domain", "visits"]].to_string(index=False))

never-measured domains                    23833.0
share of all visits (%)                       8.8
tracker-infrastructure domains (%)            3.8
tracker-infrastructure visit share (%)       24.4

largest tracker-infrastructure 'visited' domains:
        private_domain  visits
         impactcdn.com    7459
           trysera.com    6068
         yellowblue.io    5151
minutemedia-prebid.com    4707
            aaxads.com    4110
    ebayadservices.com    3873
     eu-1-id5-sync.com    3014
     contentsquare.net    2951


In [4]:
def alive_stats(sub, label):
    v = sub["visits"].sum()
    return {
        "subset": label,
        "n_domains": len(sub),
        "pct_of_visits": 100 * v / dom["visits"].sum(),
        "alive_share_domains_pct": 100 * sub["alive_2022"].mean(),
        "alive_share_visits_pct": 100 * sub.loc[sub["alive_2022"], "visits"].sum() / v,
    }


checked = unscanned.query("ha22 or wb_checked")  # domains with a real check
stats = pd.DataFrame([
    alive_stats(unscanned, "all BL-unscanned"),
    alive_stats(checked, "BL-unscanned with a 2022 check (HA or WB queried)"),
])
stats.round(1)

,subset,n_domains,pct_of_visits,alive_share_domains_pct,alive_share_visits_pct
0,all BL-unscanned,29996,23.6,22.5,65.9
1,BL-unscanned with a 2022 check (HA or WB queried),8417,19.4,80.2,80.2


In [5]:
tex = stats.copy()
tex["n_domains"] = tex["n_domains"].map("{:,}".format)
for c in tex.columns[2:]:
    tex[c] = tex[c].map("{:.1f}".format)
pandas_to_tex(tex, os.path.join(config.TABLES_DIR, "wb_liveness"))
print("wrote tables/wb_liveness.tex")

wrote tables/wb_liveness.tex


## Reading the results

The composition split comes first: visits to tracker-infrastructure domains
are not page visits with unknown tracking — they are artifacts of how the
meter logs redirect and sync requests, and for them the paper's zero-fill is
closer to a category correction than an undercount. The bounds in strand B
should be read with that share in mind.

For the genuine-website rest, `alive_share_visits_pct` on the checked subset
is the interpretable number:
among unscanned browsing where we could actually look for 2022 evidence, the
share that was demonstrably live. A high value says the 2025 scan failures
were mostly *unscannability*, not death — the paper's missing mass is ordinary
web browsing, and contemporaneous fills (strand B) are the appropriate remedy.
A low value would say much of the missing mass died with no 2022 record, and
the honest response is wider bounds, not imputation. Wayback coverage gaps
mean the true alive share can only be higher than reported, not lower.